# Кандидатогенерация для категории "Услуги"

По каждому из 2 452 поисковых запросов из `benchmark_queries.parquet` нужно отобрать до 50 объявлений из корпуса `benchmark_items.parquet` (189 212 объявлений). Качество считается по Recall@50.

**Итог: Recall@50 = 0.8837 на лидерборде.**

## Как устроено решение

1. **Пул по локации.** В train 83% выбранных объявлений находятся в той же локации, где искал пользователь. Остальные обычно лежат в соседних городах или в регионе, причём многие `search_location_id` являются регионами без собственных объявлений. Поэтому по train я считаю вероятность `P(локация объявления | локация поиска)` и ищу только в тех локациях, которые вместе дают 99.5% этой вероятности.
2. **Несколько источников кандидатов внутри пула.** Их результаты объединяются, в среднем получается около 1200 кандидатов на запрос:
   * BM25 по леммам (pymorphy3) по заголовку, параметрам и описанию, а также BM25 только по заголовку;
   * TF-IDF по символьным n-граммам заголовка, он помогает при опечатках и редких словоформах;
   * близость эмбеддингов модели `deepvk/USER-bge-m3`, top-1000;
   * поиск внутри двух самых вероятных для запроса подкатегорий. В больших городах нужное объявление часто стоит примерно на тысячном месте по всему пулу, но в своей подкатегории оказывается в первой сотне;
   * 15 лучших совпадений из других локаций, на случай если исполнитель находится не там, где его ищут;
   * объявления, которые в train выбирали по точно такому же запросу.
3. **Кросс-энкодер** `BAAI/bge-reranker-v2-m3` читает запрос и объявление вместе и оценивает 100 лучших кандидатов каждого запроса.
4. **Ранкер LightGBM (LambdaRank)** выбирает итоговые 50 объявлений. Он использует около 50 признаков: оценки и ранги всех источников и кросс-энкодера, вероятность локации, расстояние, вероятность подкатегории по тексту запроса, соответствие фильтрам поиска, рейтинг, число отзывов, цену и положение кандидата среди других кандидатов того же запроса.

## Проверка качества

Из train я отложил 10,000 групп "запрос, локация, фильтры". Половина из них нужна для обучения ранкера, вторая половина для проверки. Больше всего времени ушло не на модели, а на правильную сборку проверочного корпуса: локальная метрика дважды расходилась с лидербордом.

| Версия | Что было в проверочном корпусе | Val | Test |
|---|---|---|---|
| v4 | из train убраны все объявления-ответы, из-за чего ранкер решил, что объявления с кликами не бывают ответами | 0.922 | 0.785 |
| v5 | корпус бенчмарка и добавленные ответы. Добавленные составляли 7.7% кандидатов, но 92.7% правильных ответов, и ранкер научился их узнавать | 0.923 | 0.878 |
| v8 | в роли обманок все объявления train (около 318 тысяч), без признаков истории кликов, плюс кросс-энкодер | 0.847 | **0.8837** |

В финальной схеме по происхождению объявления уже нельзя угадать, ответ оно или нет. Корпус получился больше и сложнее настоящего, поэтому val занижен, зато честно сравнивает разные идеи между собой. Признаки истории кликов (`item_pop`, `cc_q_item`) пришлось убрать ведь у всех обманок в train есть клики, а у ответов только у 42%, и ранкер снова научился бы штрафовать популярные объявления.

## Модели и библиотеки

Все модели open-source и работают локально, внешние API не используются.
* [`deepvk/USER-bge-m3`](https://huggingface.co/deepvk/USER-bge-m3): энкодер предложений для русского языка, лицензия Apache-2.0.
* [`BAAI/bge-reranker-v2-m3`](https://huggingface.co/BAAI/bge-reranker-v2-m3): многоязычный кросс-энкодер, лицензия Apache-2.0.
* Веса скачиваются с HuggingFace в папку `models/` один раз, если их там нет. После этого всё работает без интернета.
* Библиотеки: `sentence-transformers`, `transformers`, `torch`, `pymorphy3`, `scikit-learn`, `lightgbm`, `pandas`, `numpy`, `scipy`, `pyarrow`.

## Запуск

Окружение conda `dl`, Jupyter Lab. Нужна видеокарта с 8 ГБ памяти (запускал на RTX 3070 Ti) и 32 ГБ оперативной памяти. Полный прогон с нуля занимает около трёх часов, из них примерно 35 минут уходит на эмбеддинги и около полутора часов на кросс-энкодер.

Каждый этап записан в отдельный файл `src/*.py` (ячейки `%%writefile`) и запускается отдельным процессом. Так память освобождается после каждого этапа. Промежуточные результаты сохраняются в папку `cache_final`. Если прогон прервался, при повторном запуске готовые этапы пропускаются, а кросс-энкодер продолжает считать с последней контрольной точки.

In [1]:
import os, sys, subprocess, time

CACHE_DIR = os.environ.get('CACHE_DIR', 'cache_final')
os.makedirs('src', exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs('models', exist_ok=True)


def run(script, **env):
    # запуск этапа отдельным процессом того же Python
    e = dict(os.environ, CACHE_DIR=CACHE_DIR, PYTHONIOENCODING='utf-8', **{k: str(v) for k, v in env.items()})
    t0 = time.time()
    p = subprocess.Popen([sys.executable, '-u', script], env=e, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding='utf-8')
    for line in p.stdout:
        if 'Warning' not in line and 'warn(' not in line and 'it/s]' not in line:
            print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{script} завершился с ошибкой')
    print(f'--- {script}: {time.time() - t0:.0f} с')


print('python:', sys.executable, '| кэш:', CACHE_DIR)

python: C:\Users\Alexander\.conda\envs\dl\python.exe | кэш: cache_final


## 0. Общие функции

Здесь собраны функции, которые нужны сразу нескольким этапам.

* BM25 считается через разреженные матрицы по формуле `score = Q @ W^T`, где `W[d, t]` это вес термина в документе. Так скоры сразу получаются для пачки запросов по всему корпусу, и отдельный поисковый движок не нужен.
* `LocationPool` отвечает за модель локаций из первого пункта схемы.

In [2]:
%%writefile src/common.py

import os, re, numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

CACHE = os.environ.get('CACHE_DIR', 'cache_final')


def bm25_matrix(docs, vocab=None, k1=1.2, b=0.75):
    # W[d, t]: вес BM25 термина t в документе d
    cv = CountVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False, vocabulary=vocab, dtype=np.float32)
    tf = cv.fit_transform(docs).tocsr().astype(np.float32)
    dl = np.asarray(tf.sum(1)).ravel()
    avg = dl.mean() if dl.mean() > 0 else 1.0
    df = np.bincount(tf.indices, minlength=tf.shape[1])
    n = tf.shape[0]
    idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
    rows = np.repeat(np.arange(n), np.diff(tf.indptr))
    tf.data = (tf.data * (k1 + 1) / (tf.data + k1 * (1 - b + b * dl[rows] / avg)) * idf[tf.indices]).astype(np.float32)
    return tf, cv


def query_matrix(queries, cv):
    # бинарная матрица запрос на термин
    q = cv.transform(queries).tocsr()
    q.data[:] = 1
    return q.astype(np.float32)


class LocationPool:
    def __init__(self, fit, items, mass):
        cnt = fit.groupby(['search_location_id', 'item_location_id']).size().rename('n').reset_index()
        cnt['p'] = cnt.n / cnt.groupby('search_location_id').n.transform('sum')
        self.prob = {(a, b): p for a, b, p in zip(cnt.search_location_id, cnt.item_location_id, cnt.p)}
        self.cnt = cnt.sort_values(['search_location_id', 'p'], ascending=[True, False])
        self.mass, self.item_loc, self._cache = mass, items.item_location_id.values, {}

    def mask(self, sloc):
        # булева маска объявлений, которые попадают в пул данной локации поиска
        if sloc not in self._cache:
            c = self.cnt[self.cnt.search_location_id == sloc]
            keep = set(c[(c.p.cumsum() - c.p) < self.mass].item_location_id) | {sloc}
            self._cache[sloc] = np.isin(self.item_loc, list(keep))
        return self._cache[sloc]

    def logp(self, sloc, idx):
        # логарифм вероятности локации для каждого кандидата, используется как признак ранкера
        return np.log(np.array([self.prob.get((sloc, l), 0.0) for l in self.item_loc[idx]]) + 1e-5)


def topk_masked(scores, mask, k, positive=True):
    # индексы k лучших по скору объявлений среди разрешённых маской (по умолчанию только с положительным скором)
    s = np.where(mask, scores, -np.inf)
    k = min(k, int(mask.sum()))
    if k == 0:
        return np.array([], dtype=int)
    idx = np.argpartition(-s, k - 1)[:k]
    ok = np.isfinite(s[idx]) & ((s[idx] > 0) if positive else True)
    return idx[ok]


KEYS = ('Вид услуги|Тип услуги|Кто оказывает услуги|Где вы оказываете услуги|Ваши клиенты|Онлайн-запись|'
        'Рейтинг пользователя|Специальность или сфера|Груз|Грузоподъёмность|Чем занимается исполнитель|Марка авто|'
        'Услуги|Услуга|Срочная услуга|Рабочие дни|Занятия|Участие в пилоте|Слова в описании')


def pvals(s, key):
    return re.findall(re.escape(key) + r' (.+?)(?= (?:' + KEYS + r')|$)', s)


def item_text(t, p, d):
    # текст объявления (заголовок, начало параметров и начало описания)
    return f"{t}. {(p or '')[:250]}. {(d or '')[:400]}"

Overwriting src/common.py


## 1. Отложенная выборка, проверочный корпус и лемматизация

- **Отложенные группы.** Тексты запросов выбираются равномерно среди уникальных текстов, потому что бенчмарк, судя по всему, собран так же: 37% его запросов встречаются в train, у моей выборки таких 40%. Берутся только локации, которые есть в бенчмарке. Из train удаляются только сами отложенные группы, а не объявления-ответы целиком. Иначе ранкер учится штрафовать популярные объявления (версия v4 получила из-за этого 0.785).
- **Проверочный корпус** состоит из корпуса бенчмарка, ответов отложенных групп и всех остальных объявлений train в роли обманок. Запросы бенчмарка при этом видят только настоящий корпус (флаг `in_bench`).
- **Лемматизация** выполняется через pymorphy3. Разбор кэшируется по уникальным словам, поэтому полмиллиона объявлений обрабатываются за несколько минут.

In [3]:
%%writefile src/s1_prep.py

import os, re, sys, numpy as np, pandas as pd
from pymorphy3 import MorphAnalyzer
sys.path.insert(0, 'src')
from common import CACHE

RNG = np.random.default_rng(42)
if all(os.path.exists(f'{CACHE}/{f}') for f in ('items.parquet', 'query_lemmas.parquet', 'fit.parquet', 'hold_queries.parquet')):
    print('skip')
    sys.exit(0)

tr = pd.read_parquet('dataset/train.parquet')
bq = pd.read_parquet('dataset/benchmark_queries.parquet')
bi = pd.read_parquet('dataset/benchmark_items.parquet')
for d in (tr, bq):
    d['search_infm_params_text'] = d['search_infm_params_text'].fillna('')
    d['search_query'] = d['search_query'].fillna('').str.strip().str.lower()

# отложенные группы (запрос, локация, фильтры) равномерно по уникальным текстам и только в локациях бенчмарка
tr['gkey'] = tr.search_query + '|' + tr.search_location_id.astype(str) + '|' + tr.search_infm_params_text
groups = tr[tr.search_location_id.isin(set(bq.search_location_id))].drop_duplicates('gkey')[['gkey', 'search_query']]
texts = groups.search_query.drop_duplicates().sample(frac=1, random_state=42)
hold = groups[groups.search_query.isin(set(texts.head(10000)))].sample(frac=1, random_state=1).drop_duplicates('search_query')
is_hold = tr.gkey.isin(set(hold.gkey))
hrows = tr[is_hold]
# на оставшейся части train считаются все статистики
fit = tr[~is_hold].reset_index(drop=True)
print('строк fit', len(fit), '| отложенных групп', len(hold), '| доля их запросов, встречающихся в fit:',
      round(hold.search_query.isin(set(fit.search_query)).mean(), 3))

qcols = ['search_query', 'search_location_id', 'search_is_delivery_search', 'search_infm_params_text', 'search_category']
hq = hrows.groupby('gkey').agg({**{c: 'first' for c in qcols}, 'item_id': lambda s: sorted(set(s))}).reset_index()
hq = hq.rename(columns={'item_id': 'targets'})
hq['query_id'] = ['h%015d' % i for i in range(len(hq))]
# rk нужна для обучения ранкера, val для проверки
hq['part'] = np.where(RNG.random(len(hq)) < 0.5, 'rk', 'val')
hq.drop(columns='gkey').to_parquet(f'{CACHE}/hold_queries.parquet')
fit.drop(columns='gkey').to_parquet(f'{CACHE}/fit.parquet')

# все объявления: корпус бенчмарка, ответы отложенных групп и остальные объявления train в роли обманок
icols = list(bi.columns)
extra = hrows.drop_duplicates('item_id')[icols]
extra = extra[~extra.item_id.isin(set(bi.item_id))]
decoy = fit.drop_duplicates('item_id')[icols]
decoy = decoy[~decoy.item_id.isin(set(bi.item_id) | set(extra.item_id))]
items = pd.concat([bi.assign(in_bench=True, decoy=False), extra.assign(in_bench=False, decoy=False),
                   decoy.assign(in_bench=False, decoy=True)], ignore_index=True)
for c in ['item_price', 'item_latitude', 'item_longitude']:
    items[c] = pd.to_numeric(items[c], errors='coerce')
for c in ['item_title_raw', 'item_description_raw', 'item_infm_params_text']:
    items[c] = items[c].fillna('')
print('объявлений', len(items), '| только из отложенных', len(extra), '| обманок', len(decoy))

# лемматизация с кэшем по уникальным словам
morph = MorphAnalyzer()
TOK = re.compile(r'[a-zа-яё0-9]+')
cache = {}


def lemmas(text):
    out = []
    for t in TOK.findall(text.lower().replace('ё', 'е')):
        l = cache.get(t)
        if l is None:
            l = morph.parse(t)[0].normal_form if re.search('[а-я]', t) else t
            cache[t] = l
        out.append(l)
    return ' '.join(out)


def params_clean(s):
    # служебные поля вроде рабочих дней и времени для связи повторяются почти везде и только мешают BM25
    return re.sub(r'Рабочие дни \S+|Время для связи[^А-Я]*|Тип стоимости за \S+', ' ', s)


items['lem_title'] = [lemmas(t) for t in items.item_title_raw]
items['lem_params'] = [lemmas(params_clean(t)) for t in items.item_infm_params_text]
items['lem_desc'] = [lemmas(t[:1500]) for t in items.item_description_raw]
items.to_parquet(f'{CACHE}/items.parquet')
allq = pd.concat([fit[['search_query']], hq[['search_query']], bq[['search_query']]]).drop_duplicates()
allq['lem_query'] = [lemmas(t) for t in allq.search_query]
allq.to_parquet(f'{CACHE}/query_lemmas.parquet')
print('слов в словаре лемматизатора', len(cache))

Overwriting src/s1_prep.py


In [4]:
run('src/s1_prep.py')

skip
--- src/s1_prep.py: 2 с


## 2. Эмбеддинги `deepvk/USER-bge-m3`

Модель используется как есть, без дообучения. Я пробовал дообучать её на кликах из train: на локальной проверке это давало прирост, но на лидерборде результат не улучшился (v2 получила 0.8795). Объявления сортируются по длине текста, чтобы в батчах было меньше паддинга, это ускоряет расчёт примерно в два раза. Считается в fp16 на видеокарте.

In [5]:
%%writefile src/s2_encode.py

import os, sys, numpy as np, pandas as pd
sys.path.insert(0, 'src')
from common import CACHE, item_text
from sentence_transformers import SentenceTransformer

MODEL = 'models/USER-bge-m3'
if not os.path.exists(f'{MODEL}/model.safetensors'):
    from huggingface_hub import snapshot_download
    snapshot_download('deepvk/USER-bge-m3', local_dir=MODEL)
m = SentenceTransformer(MODEL, device='cuda').half()
m.max_seq_length = 192

if not os.path.exists(f'{CACHE}/emb_items.npy'):
    items = pd.read_parquet(f'{CACHE}/items.parquet', columns=['item_title_raw', 'item_infm_params_text', 'item_description_raw'])
    texts = [item_text(*r) for r in zip(items.item_title_raw, items.item_infm_params_text, items.item_description_raw)]
    order = np.argsort([len(t) for t in texts])
    E = m.encode([texts[i] for i in order], batch_size=128, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    emb = np.zeros_like(E)
    emb[order] = E
    
    np.save(f'{CACHE}/emb_items_tmp.npy', emb.astype(np.float16))
    os.replace(f'{CACHE}/emb_items_tmp.npy', f'{CACHE}/emb_items.npy')
    print('посчитаны эмбеддинги объявлений:', len(items))

hq = pd.read_parquet(f'{CACHE}/hold_queries.parquet')
bq = pd.read_parquet('dataset/benchmark_queries.parquet')
qs = pd.concat([hq.search_query, bq.search_query.fillna('').str.strip().str.lower()]).drop_duplicates().tolist()
np.save(f'{CACHE}/emb_q_tmp.npy', m.encode(qs, batch_size=512, normalize_embeddings=True, convert_to_numpy=True).astype(np.float16))
os.replace(f'{CACHE}/emb_q_tmp.npy', f'{CACHE}/emb_q.npy')
pd.Series(qs).to_frame('q').to_parquet(f'{CACHE}/emb_q_keys.parquet')
print('посчитаны эмбеддинги запросов:', len(qs))

Overwriting src/s2_encode.py


In [6]:
run('src/s2_encode.py')

посчитаны эмбеддинги запросов: 12316


--- src/s2_encode.py: 20 с


## 3. Кандидаты и признаки

Для каждого запроса внутри его локационного пула объединяются результаты всех источников:
* BM25: 300 лучших;
* BM25 по заголовку и символьные n-граммы: по 80 лучших;
* эмбеддинги: 1000 лучших;
* по 100 объявлений внутри двух самых вероятных подкатегорий;
* 15 объявлений из других локаций;
* co-click.

Для каждой пары "запрос, кандидат" считаются признаки. Ранги и относительные скоры считаются относительно всего пула запроса, поэтому признаки сравнимы между короткими и длинными запросами. Кандидатов получается около 15 миллионов строк, и держать их все в памяти рискованно, поэтому они записываются на диск частями примерно по 1 300 запросов.

In [7]:
%%writefile src/s3_cands.py

import os, sys, gc, time, shutil, numpy as np, pandas as pd, torch
sys.path.insert(0, 'src')
from common import *
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

K_BM, K_TITLE, K_CHAR, K_DENSE = 300, 80, 80, 1000
K_MC, MC_TOP, MC_MIN_P = 100, 2, 0.1   # поиск внутри подкатегорий по 100 объявлений в 2 лучших подкатегориях
K_GLOBAL = 15                          # сколько объявлений брать из других локаций
POOL_MASS, B = 0.995, 128              # доля вероятности в пуле локаций и размер пачки запросов
if os.path.exists(f'{CACHE}/cands/_done') and os.path.exists(f'{CACHE}/Qdf.parquet'):
    print('этап 3 уже выполнен, skip')
    sys.exit(0)

items = pd.read_parquet(f'{CACHE}/items.parquet')
fit = pd.read_parquet(f'{CACHE}/fit.parquet')
hq = pd.read_parquet(f'{CACHE}/hold_queries.parquet')
bq = pd.read_parquet('dataset/benchmark_queries.parquet')
bq['search_query'] = bq.search_query.fillna('').str.strip().str.lower()
bq['search_infm_params_text'] = bq.search_infm_params_text.fillna('')
bq['targets'] = [[] for _ in range(len(bq))]
bq['part'] = 'bench'
Qdf = pd.concat([hq, bq[hq.columns]], ignore_index=True)
ql = pd.read_parquet(f'{CACHE}/query_lemmas.parquet').set_index('search_query').lem_query
Qdf['lem'] = Qdf.search_query.map(ql).fillna('')
fit['lem'] = fit.search_query.map(ql).fillna('')
N = len(items)
ids = items.item_id.values
id2idx = pd.Series(np.arange(N), index=ids)
bench_mask = items.in_bench.values

# лексические индексы
W_all, cv = bm25_matrix(((items.lem_title + ' ') * 3 + items.lem_params + ' ' + items.lem_desc).tolist())
W_title, _ = bm25_matrix(items.lem_title.tolist(), vocab=cv.vocabulary_)
W_par, _ = bm25_matrix(items.lem_params.tolist(), vocab=cv.vocabulary_)
W_desc, _ = bm25_matrix(items.lem_desc.tolist(), vocab=cv.vocabulary_)
W_all_t, W_title_t, W_par_t, W_desc_t = [w.T.tocsr() for w in (W_all, W_title, W_par, W_desc)]
Qm = query_matrix(Qdf.lem.tolist(), cv)
ctf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4), min_df=3, sublinear_tf=True, dtype=np.float32)
C_items = ctf.fit_transform(items.item_title_raw.str.lower()).T.tocsr()
C_q = ctf.transform(Qdf.search_query)

# эмбеддинги
E = torch.tensor(np.load(f'{CACHE}/emb_items.npy')).cuda()
qk = pd.read_parquet(f'{CACHE}/emb_q_keys.parquet').q
QE = torch.tensor(np.load(f'{CACHE}/emb_q.npy')[pd.Series(np.arange(len(qk)), index=qk)[Qdf.search_query].values]).cuda()

# статистики, посчитанные только по fit
lp = LocationPool(fit, items, POOL_MASS)
# вероятность подкатегории по тексту запроса наивный байес по леммам и парам лемм
mc_codes, mc_uni = pd.factorize(fit.item_microcat_id)
nb_vec = TfidfVectorizer(tokenizer=str.split, token_pattern=None, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
nb = MultinomialNB(alpha=0.05).fit(nb_vec.fit_transform(fit.lem), mc_codes)
item_mc_col = pd.Series(np.arange(len(mc_uni)), index=mc_uni).reindex(items.item_microcat_id.values).fillna(-1).astype(int).values
# co-click сколько раз объявление выбирали по такому же запросу
cc_q = {q: g.droplevel(0) for q, g in fit.groupby(['search_query', 'item_id']).size().groupby(level=0)}
items_pop = fit.item_id.value_counts().reindex(ids).fillna(0).values
# как часто подкатегорию выбирают в данной локации
mc_loc = fit.groupby(['search_location_id', 'item_microcat_id']).size()
iparams, rating = items.item_infm_params_text.values, items.item_rating.values
lat, lon = items.item_latitude.values, items.item_longitude.values
cent = pd.concat([fit[['item_location_id']].assign(lat=pd.to_numeric(fit.item_latitude, errors='coerce'),
                                                   lon=pd.to_numeric(fit.item_longitude, errors='coerce')),
                  items[['item_location_id']].assign(lat=lat, lon=lon)]).groupby('item_location_id')[['lat', 'lon']].median()

title_len = items.item_title_raw.str.len().values
items = items[['item_id', 'item_location_id', 'item_microcat_id', 'item_rating_reviews_count', 'item_price']].copy()
del fit, W_all, W_title, W_par, W_desc
gc.collect()

OUT = f'{CACHE}/cands'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)
tgt = {i: set(id2idx.reindex(t).dropna().astype(int)) for i, t in enumerate(Qdf.targets)}
parts, stats = [0], []


def flush(rows):
    F = pd.concat(rows, ignore_index=True)
    F['rating'] = rating[F.item_idx]
    F['n_reviews'] = items.item_rating_reviews_count.values[F.item_idx]
    F['price'] = np.log1p(items.item_price.values[F.item_idx].astype(float).clip(0))
    F['microcat'] = items.item_microcat_id.values[F.item_idx]
    F['title_len'] = title_len[F.item_idx]
    F['q_len'] = Qdf.search_query.str.split().str.len().values[F.qi]
    F['q_cat'] = Qdf.search_category.values[F.qi]
    F['q_has_params'] = (Qdf.search_infm_params_text.values[F.qi] != '').astype(np.int8)
    F['n_cand'] = F.groupby('qi').item_idx.transform('size')
    F['y'] = np.fromiter((ii in tgt[qi] for qi, ii in zip(F.qi, F.item_idx)), dtype=np.int8, count=len(F))
    fc = F.select_dtypes('float64').columns
    F[fc] = F[fc].astype(np.float32)
    F.to_parquet(f'{OUT}/part_{parts[0]:03d}.parquet')
    parts[0] += 1
    stats.append(F.groupby('qi').agg(hits=('y', 'sum'), n=('y', 'size')))


rows, t0 = [], time.time()
for s0 in range(0, len(Qdf), B):
    sl = slice(s0, min(s0 + B, len(Qdf)))
    q = Qm[sl]
    S_all, S_title = (q @ W_all_t).toarray(), (q @ W_title_t).toarray()
    S_par, S_desc = (q @ W_par_t).toarray(), (q @ W_desc_t).toarray()
    S_char = (C_q[sl] @ C_items).toarray()
    S_den = (QE[sl] @ E.T).float().cpu().numpy()
    mc_prob = nb.predict_proba(nb_vec.transform(Qdf.lem.values[sl]))
    for j in range(S_all.shape[0]):
        r = Qdf.iloc[s0 + j]
        base = bench_mask if r.part == 'bench' else np.ones(N, bool)
        pool = lp.mask(r.search_location_id) & base
        cand = set(topk_masked(S_all[j], pool, K_BM)) | set(topk_masked(S_title[j], pool, K_TITLE)) \
            | set(topk_masked(S_char[j], pool, K_CHAR)) | set(topk_masked(S_den[j], pool, K_DENSE, positive=False))
        # поиск внутри самых вероятных подкатегорий
        for m in np.argsort(-mc_prob[j])[:MC_TOP]:
            if mc_prob[j][m] < MC_MIN_P:
                break
            cand |= set(topk_masked(S_den[j], pool & (item_mc_col == m), K_MC, positive=False))
        # сильные совпадения из других локаций
        cand |= set(topk_masked(S_den[j], base & ~pool, K_GLOBAL, positive=False))
        cc = cc_q.get(r.search_query)
        if cc is not None:
            ci = id2idx.reindex(cc.index).dropna().astype(int).values
            cand |= set(ci[base[ci]])
        c = np.fromiter(cand, dtype=int)
        if len(c) == 0:
            continue
        f = pd.DataFrame({'qi': s0 + j, 'item_idx': c, 'bm_all': S_all[j, c], 'bm_title': S_title[j, c],
                          'bm_par': S_par[j, c], 'bm_desc': S_desc[j, c], 'char': S_char[j, c], 'dense': S_den[j, c]})
        # ранг внутри пула и скор относительно лучшего в пуле
        for col, sc in (('bm_all', S_all[j]), ('char', S_char[j]), ('dense', S_den[j])):
            pv = np.sort(sc[pool])
            f[col + '_rk'] = len(pv) - np.searchsorted(pv, sc[c], side='right')
            mx = pv[-1] if len(pv) else 0
            f[col + '_rel'] = sc[c] - mx if col == 'dense' else sc[c] / (mx + 1e-6)
        f['in_pool'] = pool[c].astype(np.int8)
        f['loc_logp'] = lp.logp(r.search_location_id, c)
        f['same_loc'] = (items.item_location_id.values[c] == r.search_location_id).astype(np.int8)
        if r.search_location_id in cent.index:
            clat, clon = cent.loc[r.search_location_id]
            f['dist'] = np.hypot(lat[c] - clat, (lon[c] - clon) * np.cos(np.radians(clat)))
        else:
            f['dist'] = np.nan
        mcc = item_mc_col[c]
        f['mc_prob'] = np.where(mcc >= 0, mc_prob[j][np.maximum(mcc, 0)], 0)
        f['mc_loc_pop'] = mc_loc.reindex(list(zip([r.search_location_id] * len(c), items.item_microcat_id.values[c]))).fillna(0).values
        f['cc_q_item'] = cc.reindex(ids[c]).fillna(0).values if cc is not None else 0
        f['item_pop'] = items_pop[c]
        # соответствие фильтрам поиска ( -1 => фильтр не задан)
        p = r.search_infm_params_text
        for key, name in (('Вид услуги', 'f_vid'), ('Тип услуги', 'f_tip')):
            vals = [v for v in pvals(p, key) if v and not v.startswith('автосервиса')]
            f[name] = np.array([any((key + ' ' + v) in iparams[i] for v in vals) for i in c], np.int8) if vals else -1
        f['f_rating'] = (np.nan_to_num(rating[c]) >= 4).astype(np.int8) if 'Рейтинг пользователя' in p else -1
        rows.append(f)
    if (s0 // B) % 10 == 9 or s0 + B >= len(Qdf):
        flush(rows)
        rows = []
        print(f'{min(s0 + B, len(Qdf))}/{len(Qdf)} запросов, {time.time() - t0:.0f} с', flush=True)

Qdf.drop(columns=['lem']).to_parquet(f'{CACHE}/Qdf.parquet')
open(f'{OUT}/_done', 'w').write('ok') 
S = pd.concat(stats)
for part in ['rk', 'val']:
    qs = np.where(Qdf.part == part)[0]
    print(part, 'recall кандидатов (потолок для ранкера)',
          round((S.hits.reindex(qs).fillna(0) / Qdf.targets.str.len().values[qs]).mean(), 4),
          '| кандидатов на запрос в среднем', round(S.n.reindex(qs).mean()))

Overwriting src/s3_cands.py


In [8]:
run('src/s3_cands.py')

этап 3 уже выполнен, skip


--- src/s3_cands.py: 4 с


## 4. Кросс-энкодер `BAAI/bge-reranker-v2-m3`

Сначала предварительный LightGBM выбирает 100 лучших кандидатов для каждого запроса. Он обучается крест-накрест: модель, обученная на `rk`, выбирает кандидатов для `val` и бенчмарка, а модель, обученная на `val`, выбирает для `rk`. Так ни одна часть данных не отбирается моделью, которая видела её разметку.

Затем кросс-энкодер оценивает около 1.25 миллиона пар. Размер батча 32: при 128 модель не помещается в 8 GB VRAM, и скорость падает примерно в пять раз. Прогресс сохраняется каждые 25 тысяч пар, поэтому после прерывания расчёт продолжается с того же места.

In [9]:
%%writefile src/s4_ce.py

import os, sys, gc, time, numpy as np, pandas as pd, torch
sys.path.insert(0, 'src')
from common import CACHE

TOPN, BS, CKPT = 100, 32, 800
DROP = ['item_pop', 'cc_q_item']        # те же признаки исключены и у финального ранкера
PAIRS, PARTIAL, POS = f'{CACHE}/ce_pairs.parquet', f'{CACHE}/ce_partial.npy', f'{CACHE}/ce_pos.txt'
MODEL = 'models/bge-reranker-v2-m3'
if os.path.exists(f'{CACHE}/ce.parquet'):
    print('этап 4 уже выполнен, пропускаю')
    sys.exit(0)
Q = pd.read_parquet(f'{CACHE}/Qdf.parquet', columns=['search_query', 'part'])

if not os.path.exists(PAIRS):
    import lightgbm as lgb
    F = pd.read_parquet(f'{CACHE}/cands')
    F['part'] = Q.part.values[F.qi]
    feats = [c for c in F.columns if c not in ('qi', 'item_idx', 'y', 'part') and c not in DROP]

    def fit(tr):
        tr = tr[tr.groupby('qi').y.transform('max') > 0].sort_values('qi')
        return lgb.train(dict(objective='lambdarank', learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
                              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                              lambdarank_truncation_level=100, verbose=-1),
                         lgb.Dataset(tr[feats], tr.y, group=tr.groupby('qi').size().values,
                                     categorical_feature=['microcat', 'q_cat']), 600)

    F['s1'] = np.float32(0)
    m = fit(F[F.part == 'val'])
    F.loc[F.part == 'rk', 's1'] = m.predict(F.loc[F.part == 'rk', feats])
    del m; gc.collect()
    m = fit(F[F.part == 'rk'])
    F.loc[F.part != 'rk', 's1'] = m.predict(F.loc[F.part != 'rk', feats])
    del m; gc.collect()
    top = F.sort_values(['qi', 's1'], ascending=[True, False]).groupby('qi').head(TOPN)[['qi', 'item_idx']]
    top.reset_index(drop=True).to_parquet(PAIRS)
    del F, top; gc.collect()
top = pd.read_parquet(PAIRS)
print('пар для оценки', len(top), flush=True)

need = np.unique(top.item_idx.values)
items = pd.read_parquet(f'{CACHE}/items.parquet', columns=['item_title_raw', 'item_infm_params_text', 'item_description_raw']).iloc[need]
doc = dict(zip(need, (items.item_title_raw + '. ' + items.item_infm_params_text.str[:200] + '. '
                      + items.item_description_raw.str[:300]).values))
del items; gc.collect()
qtext = Q.search_query.values

if not os.path.exists(f'{MODEL}/model.safetensors'):
    from huggingface_hub import snapshot_download
    snapshot_download('BAAI/bge-reranker-v2-m3', local_dir=MODEL)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, torch_dtype=torch.float16).cuda().eval()
# пары сортируются по длине текста, чтобы в батчах было меньше паддинга
order = np.argsort([len(doc[i]) for i in top.item_idx.values], kind='stable')
scores = np.load(PARTIAL) if os.path.exists(PARTIAL) else np.zeros(len(top), np.float32)
start = int(open(POS).read()) if os.path.exists(POS) else 0
qi_v, it_v, t0 = top.qi.values, top.item_idx.values, time.time()
with torch.inference_mode():
    for b, s in enumerate(range(start, len(order), BS)):
        idx = order[s:s + BS]
        enc = tok([qtext[q] for q in qi_v[idx]], [doc[i] for i in it_v[idx]], padding=True,
                  truncation='only_second', max_length=256, return_tensors='pt').to('cuda')
        scores[idx] = model(**enc).logits.float().squeeze(-1).cpu().numpy()
        if (b + 1) % CKPT == 0 or s + BS >= len(order):
            np.save(PARTIAL.replace('.npy', '_tmp.npy'), scores)
            os.replace(PARTIAL.replace('.npy', '_tmp.npy'), PARTIAL)
            open(POS, 'w').write(str(s + BS))
            if (b + 1) % (CKPT * 8) == 0:
                print(f'{s + BS}/{len(order)} пар, {time.time() - t0:.0f} с', flush=True)
top['ce'] = scores
top.to_parquet(f'{CACHE}/ce.parquet')
print('сохранено оценок:', len(top))

Overwriting src/s4_ce.py


In [10]:
run('src/s4_ce.py')

этап 4 уже выполнен, пропускаю


--- src/s4_ce.py: 4 с


## 5. Ранкер LightGBM и файл `answer.csv`

* Добавляются признаки положения кандидата внутри своего запроса: z-оценка и перцентиль скоров, ранг внутри подкатегории. Градиентный бустинг смотрит на каждую строку отдельно и сам сравнивать кандидата с соседями не умеет.
* Из кросс-энкодера берутся три признака: сама оценка, ранг в запросе и отставание от лучшего кандидата запроса.
* Признаки `item_pop` и `cc_q_item` не используются, причина описана в разделе о проверке качества.
* Модель обучается на `rk` и проверяется на `val`. Финальная модель обучается на `rk` и `val` вместе и применяется к бенчмарку.
* Если у запроса оказалось меньше 50 кандидатов, список дополняется популярными объявлениями из его пула. Лишние id Recall не уменьшают.
* В конце выполняется проверка файла на все требования задания.

In [11]:
%%writefile src/s5_rank.py

import os, sys, re, numpy as np, pandas as pd, lightgbm as lgb
sys.path.insert(0, 'src')
from common import CACHE, LocationPool

DROP = ['item_pop', 'cc_q_item']
F = pd.read_parquet(f'{CACHE}/cands')
Q = pd.read_parquet(f'{CACHE}/Qdf.parquet')
F['part'] = Q.part.values[F.qi]
# у кандидатов вне первой сотни оценки кросс-энкодера нет, там остаётся NaN
F = F.merge(pd.read_parquet(f'{CACHE}/ce.parquet'), on=['qi', 'item_idx'], how='left')
F['ce_rk'] = F.groupby('qi').ce.rank(ascending=False)
F['ce_rel'] = F.ce - F.groupby('qi').ce.transform('max')
# положение кандидата среди других кандидатов того же запроса
g = F.groupby('qi')
for c in ('dense', 'bm_all', 'bm_title', 'char', 'mc_prob', 'loc_logp', 'ce'):
    F[c + '_z'] = ((F[c] - g[c].transform('mean')) / (g[c].transform('std') + 1e-6)).astype(np.float32)
    F[c + '_pct'] = g[c].rank(pct=True).astype(np.float32)
F['dense_mc_rk'] = F.groupby(['qi', 'microcat']).dense.rank(ascending=False).astype(np.float32)
F['mc_n'] = F.groupby(['qi', 'microcat']).item_idx.transform('size').astype(np.float32)
feats = [c for c in F.columns if c not in ('qi', 'item_idx', 'y', 'part') and c not in DROP]
print('признаков', len(feats))


def fit_rank(train):
    # запросы, у которых среди кандидатов нет правильного ответа, ранжированию не учат, поэтому их убираем
    train = train[train.groupby('qi').y.transform('max') > 0].sort_values('qi')
    return lgb.train(dict(objective='lambdarank', metric='ndcg', eval_at=[50], learning_rate=0.05, num_leaves=63,
                          min_data_in_leaf=50, feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
                          lambdarank_truncation_level=100, verbose=-1),
                     lgb.Dataset(train[feats], train.y, group=train.groupby('qi').size().values,
                                 categorical_feature=['microcat', 'q_cat']), 600)


def top50(df, score):
    return df.assign(score=score).sort_values(['qi', 'score'], ascending=[True, False]).groupby('qi').head(50)


# проверка на val
val = F[F.part == 'val']
qs = np.where(Q.part == 'val')[0]
nt = Q.targets.str.len().values
cand_rec = (val.groupby('qi').y.sum().reindex(qs).fillna(0) / nt[qs]).mean()
m = fit_rank(F[F.part == 'rk'])
t = top50(val, m.predict(val[feats]))
rec = (t.groupby('qi').y.sum().reindex(qs).fillna(0) / nt[qs]).mean()
print(f'val: recall кандидатов {cand_rec:.4f} | Recall@50 {rec:.4f}')
print('самые важные признаки:', pd.Series(m.feature_importance('gain'), feats).sort_values(ascending=False).head(10).round(0).to_dict())
del val, t, m

# финальная модель и ответ
m = fit_rank(F[F.part != 'bench'])
b = F[F.part == 'bench']
pred = top50(b, m.predict(b[feats])).groupby('qi').item_idx.apply(list).to_dict()
items = pd.read_parquet(f'{CACHE}/items.parquet', columns=['item_id', 'item_location_id', 'in_bench'])
ids, bench = items.item_id.values, items.in_bench.values
fit = pd.read_parquet(f'{CACHE}/fit.parquet', columns=['search_location_id', 'item_location_id', 'item_id'])
pop = fit.item_id.value_counts().reindex(ids).fillna(0).values
lp = LocationPool(fit, items, 0.995)
bench_idx = np.where(bench)[0]
bench_by_pop = bench_idx[np.argsort(-pop[bench_idx], kind='stable')]
answers = []
for qi in np.where(Q.part == 'bench')[0]:
    lst = list(pred.get(qi, []))
    if len(lst) < 50:
        # дополняем популярными объявлениями пула, а если их не хватит, то всего корпуса
        pool = np.where(lp.mask(Q.search_location_id[qi]) & bench)[0]
        seen = set(lst)
        for i in np.concatenate([pool[np.argsort(-pop[pool], kind='stable')], bench_by_pop]):
            if len(lst) >= 50:
                break
            if i not in seen:
                lst.append(i)
                seen.add(i)
    answers.append(' '.join(ids[lst[:50]]))
pd.DataFrame({'query_id': Q.query_id.values[Q.part == 'bench'], 'answer': answers}).to_csv('answer.csv', index=False)

# проверка формата по требованиям задания
bq = pd.read_parquet('dataset/benchmark_queries.parquet')
corpus = set(pd.read_parquet('dataset/benchmark_items.parquet', columns=['item_id']).item_id)
chk = pd.read_csv('answer.csv', dtype=str, keep_default_na=False)
lists = chk.answer.str.split()
assert list(chk.columns) == ['query_id', 'answer']
assert len(chk) == len(bq) and set(chk.query_id) == set(bq.query_id) and not chk.query_id.duplicated().any()
assert chk.query_id.str.len().eq(16).all()
assert lists.map(len).eq(50).all() and all(len(set(l)) == 50 for l in lists)
assert all(x in corpus and re.fullmatch('[0-9a-f]{16}', x) for l in lists for x in l)
print('answer.csv прошёл проверку:', chk.shape, '| у каждого запроса 50 разных id из корпуса')

Overwriting src/s5_rank.py


In [12]:
run('src/s5_rank.py')

признаков 49


val: recall кандидатов 0.9139 | Recall@50 0.8481
самые важные признаки: {'ce_rel': 412523.0, 'ce': 97368.0, 'microcat': 92834.0, 'ce_rk': 70991.0, 'dist': 36457.0, 'bm_all_z': 21544.0, 'ce_z': 18762.0, 'bm_all_rel': 14365.0, 'loc_logp_pct': 14307.0, 'loc_logp': 12570.0}


answer.csv прошёл проверку: (2452, 2) | у каждого запроса 50 разных id из корпуса


--- src/s5_rank.py: 728 с


In [13]:
import pandas as pd
pd.read_csv('answer.csv').head()

,query_id,answer
0,70DfDUpwjxB4lzFd,6dfa83ed47d5bd10 4332513c8fe7e1c9 f4e8903d7689...
1,JTrdTaZJvSiLPkXj,cbeccbecb1fb8d86 1c6d2079e07f4497 422d3ffdd5bb...
2,LZCZNoVG4AFUkVRJ,10eb6d6da3230bbb d62074dd39caefee dab52187b450...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 67b2cd847669eb36 bd2cb4500fad...
4,YgHcM9MVbxKnxD1e,9c41fe789f98d702 1a62634394544781 615c73ea4c3b...
